In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [4]:
from src.configs import (S, C, B, IDX_TO_CLASS, CONFIDENCE_THRESHOLD,
                         NMS_IOU_THRESHOLD)
from src.utils import convert_xywh_coords, IoU
from operator import itemgetter

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]

                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False)

                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

def filter_group_sort_preds(decoded_preds):
    sorted_preds = []

    # 1. filter and group remaining predictions by class
    for image in decoded_preds:
        valid_preds = {}
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                class_name = pred[0]
                if class_name in valid_preds:
                    valid_preds[class_name].append(pred)
                else:
                    valid_preds[class_name] = [pred]

        sorted_preds.append(valid_preds)

    # 2. sort each class's predictions by confidence score
    for image in sorted_preds:
        for class_name in image:
            image[class_name].sort(key=itemgetter(1))

    return sorted_preds


def NMS(preds_batch):
    # 1. decode batch of predictions
    decoded_preds = decode_preds(preds_batch)

    # 2. filter, group, and sort the decoded predictions
    sorted_preds = filter_group_sort_preds(decoded_preds)

    # 3. perform Non-Maximum Suppression
    final_preds = []

    for image in sorted_preds:
        final_img_preds = {}

        for class_name, preds in image.items():
            final_img_preds[class_name] = []

            while preds:
                highest_conf = preds.pop(0)
                final_img_preds[class_name].append(highest_conf)

                preds = [pred for pred in preds if
                         IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

        final_preds.append(final_img_preds)

    return final_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [6]:
decoded_preds = decode_preds(preds)
decoded_preds

[[('aeroplane',
   0.08407749226211791,
   tensor(-28.7231, grad_fn=<SubBackward0>),
   tensor(-23.0125, grad_fn=<SubBackward0>),
   tensor(46.3883, grad_fn=<AddBackward0>),
   tensor(34.1400, grad_fn=<AddBackward0>)),
  ('aeroplane',
   0.14245531375179254,
   tensor(-8.6609, grad_fn=<SubBackward0>),
   tensor(3.7522, grad_fn=<SubBackward0>),
   tensor(4.8586, grad_fn=<AddBackward0>),
   tensor(-17.1395, grad_fn=<AddBackward0>)),
  ('bird',
   -0.08566473530755125,
   tensor(69.8003, grad_fn=<SubBackward0>),
   tensor(22.3204, grad_fn=<SubBackward0>),
   tensor(-1.8872, grad_fn=<AddBackward0>),
   tensor(-34.4500, grad_fn=<AddBackward0>)),
  ('bird',
   -0.11233272994899401,
   tensor(34.9789, grad_fn=<SubBackward0>),
   tensor(18.5913, grad_fn=<SubBackward0>),
   tensor(37.5864, grad_fn=<AddBackward0>),
   tensor(-15.9457, grad_fn=<AddBackward0>)),
  ('chair',
   0.107819476223451,
   tensor(54.4882, grad_fn=<SubBackward0>),
   tensor(-6.0359, grad_fn=<SubBackward0>),
   tensor(72.67

In [7]:
sorted_preds = filter_group_sort_preds(decoded_preds)
sorted_preds

[{'train': [('train',
    0.38282636800484937,
    tensor(159.6745, grad_fn=<SubBackward0>),
    tensor(110.9916, grad_fn=<SubBackward0>),
    tensor(172.1951, grad_fn=<AddBackward0>),
    tensor(94.8419, grad_fn=<AddBackward0>))]},
 {'bird': [('bird',
    0.4516814135624365,
    tensor(78.3804, grad_fn=<SubBackward0>),
    tensor(76.8327, grad_fn=<SubBackward0>),
    tensor(-23.1464, grad_fn=<AddBackward0>),
    tensor(58.3157, grad_fn=<AddBackward0>))],
  'dog': [('dog',
    0.4370763147034431,
    tensor(68.5505, grad_fn=<SubBackward0>),
    tensor(101.4559, grad_fn=<SubBackward0>),
    tensor(-1.2652, grad_fn=<AddBackward0>),
    tensor(92.8710, grad_fn=<AddBackward0>))],
  'boat': [('boat',
    0.5885698611135126,
    tensor(264.8705, grad_fn=<SubBackward0>),
    tensor(133.2410, grad_fn=<SubBackward0>),
    tensor(136.9771, grad_fn=<AddBackward0>),
    tensor(117.5880, grad_fn=<AddBackward0>))],
  'sofa': [('sofa',
    0.40838917136264286,
    tensor(-33.5483, grad_fn=<SubBackwar

In [8]:
import torch 
nums = torch.arange(10)
nums = nums.sort()[0]

print(nums)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [9]:
nums = [num for num in nums if num == 2 or num == 4 or num == 6]
nums

[tensor(2), tensor(4), tensor(6)]

In [10]:
final_preds = []

for image in sorted_preds:
    final_img_preds = {}

    for class_name, preds in image.items():
        final_img_preds[class_name] = []

        while preds:
            highest_conf = preds.pop(0)
            final_img_preds[class_name].append(highest_conf)

            preds = [pred for pred in preds if
                     IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

    final_preds.append(final_img_preds)

[('train', 0.38282636800484937, tensor(159.6745, grad_fn=<SubBackward0>), tensor(110.9916, grad_fn=<SubBackward0>), tensor(172.1951, grad_fn=<AddBackward0>), tensor(94.8419, grad_fn=<AddBackward0>))]
[]
[('bird', 0.4516814135624365, tensor(78.3804, grad_fn=<SubBackward0>), tensor(76.8327, grad_fn=<SubBackward0>), tensor(-23.1464, grad_fn=<AddBackward0>), tensor(58.3157, grad_fn=<AddBackward0>))]
[]
[('dog', 0.4370763147034431, tensor(68.5505, grad_fn=<SubBackward0>), tensor(101.4559, grad_fn=<SubBackward0>), tensor(-1.2652, grad_fn=<AddBackward0>), tensor(92.8710, grad_fn=<AddBackward0>))]
[]
[('boat', 0.5885698611135126, tensor(264.8705, grad_fn=<SubBackward0>), tensor(133.2410, grad_fn=<SubBackward0>), tensor(136.9771, grad_fn=<AddBackward0>), tensor(117.5880, grad_fn=<AddBackward0>))]
[]
[('sofa', 0.40838917136264286, tensor(-33.5483, grad_fn=<SubBackward0>), tensor(162.0557, grad_fn=<SubBackward0>), tensor(14.4616, grad_fn=<AddBackward0>), tensor(182.2744, grad_fn=<AddBackward0>))]

In [77]:
final_preds, len(final_preds)

([{'train': [('train',
     0.38282636800484937,
     tensor(159.6745, grad_fn=<SubBackward0>),
     tensor(110.9916, grad_fn=<SubBackward0>),
     tensor(172.1951, grad_fn=<AddBackward0>),
     tensor(94.8419, grad_fn=<AddBackward0>))]},
  {'bird': [('bird',
     0.4516814135624365,
     tensor(78.3804, grad_fn=<SubBackward0>),
     tensor(76.8327, grad_fn=<SubBackward0>),
     tensor(-23.1464, grad_fn=<AddBackward0>),
     tensor(58.3157, grad_fn=<AddBackward0>))],
   'dog': [('dog',
     0.4370763147034431,
     tensor(68.5505, grad_fn=<SubBackward0>),
     tensor(101.4559, grad_fn=<SubBackward0>),
     tensor(-1.2652, grad_fn=<AddBackward0>),
     tensor(92.8710, grad_fn=<AddBackward0>))],
   'boat': [('boat',
     0.5885698611135126,
     tensor(264.8705, grad_fn=<SubBackward0>),
     tensor(133.2410, grad_fn=<SubBackward0>),
     tensor(136.9771, grad_fn=<AddBackward0>),
     tensor(117.5880, grad_fn=<AddBackward0>))],
   'sofa': [('sofa',
     0.40838917136264286,
     tensor(-3

In [17]:
y_batch.shape, y_batch, y_batch[0].shape

(torch.Size([32, 7, 7, 30]),
 tensor([[[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
 
          [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
 
          [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.00

In [57]:
(y_batch[0].flatten(0, 1))[29]

tensor([0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0625, 0.8750, 0.0938, 0.1652, 1.0000, 0.0625, 0.8750,
        0.0938, 0.1652, 1.0000])

In [74]:
from src.configs import IDX_TO_CLASS

def find_objects(y_batch):
    y_batch = y_batch.flatten(1, 2)
    truth_objects = []

    for target in y_batch:
        class_objects = {}
        
        for cell in target:
            class_name = IDX_TO_CLASS[int(torch.argmax(cell[:C]))]

            if cell[C+4] == 1:
                if class_name in class_objects:
                    class_objects[class_name].append(cell[C:C+10])
                else:
                    class_objects[class_name] = [cell[C:C+10]]

        truth_objects.append(class_objects)
        
    return truth_objects

truth_objects = find_objects(y_batch)

In [76]:
len(truth_objects)

32

In [80]:
final_preds[0]

{'train': [('train',
   0.38282636800484937,
   tensor(159.6745, grad_fn=<SubBackward0>),
   tensor(110.9916, grad_fn=<SubBackward0>),
   tensor(172.1951, grad_fn=<AddBackward0>),
   tensor(94.8419, grad_fn=<AddBackward0>))]}

In [84]:
truth_objects[0]

{'bird': [tensor([0.0625, 0.8750, 0.0938, 0.1652, 1.0000, 0.0625, 0.8750, 0.0938, 0.1652,
          1.0000])]}

In [ ]:
TP_IOU_THRESHOLD = 0.5

def find_tp_fp(final_preds, truth_objects):
    batch_tp_fp = []

    for b in range(len(final_preds)):
        class_tp_fp = {}
        
        preds = final_preds[b]
        objects = truth_objects[b]

        for class_name, target_list in objects.items():
            class_tp_fp[class_name] = []

            for target in target_list:
                
            
        
        

        